# Gold Belga Press Articles

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
#import seaborn as sns

In [5]:
gold_belga_press_articles_df = pd.read_csv('../raw/gold_belga_press_articles.csv', sep=',', on_bad_lines='skip')
gold_belga_press_articles_df.info()
gold_belga_press_articles_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48677 entries, 0 to 48676
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   article_id     48677 non-null  int64  
 1   date           48677 non-null  object 
 2   title          48677 non-null  object 
 3   lead           48667 non-null  object 
 4   source         48677 non-null  object 
 5   type           48677 non-null  object 
 6   match_id       8592 non-null   object 
 7   days_to_match  8592 non-null   float64
dtypes: float64(1), int64(1), object(6)
memory usage: 3.0+ MB


,article_id,date,title,lead,source,type,match_id,days_to_match
0,16759,2023-01-05,Al 10 eersteklassers in buitenlandse handen en...,che stem hebben.Hoofdaandeelhouder: Randy Fran...,Het Laatste Nieuws,NEWS,NaN,NaN
1,16760,2023-01-05,Football Talk. Ravych (Cercle Brugge) valt voo...,...edwensen van kinderen vervuld worden. Daarn...,Het Laatste Nieuws,NEWS,NaN,NaN
2,16761,2023-01-05,Football Talk. Vandaag nog geen debuut voor Ro...,...edwensen van kinderen vervuld worden. Daarn...,Het Laatste Nieuws,NEWS,NaN,NaN
3,16762,2023-01-05,Jonas Bogaerts en KVK Tienen willen goede reek...,de overwinning tegen OHL betekende ook de eers...,Het Laatste Nieuws,NEWS,NaN,NaN
4,16763,2023-01-05,Mathieu Maertens (OHL): “Sportieve revanche te...,OHL ontvangt zondagavond Kortrijk aan Den Dree...,Het Laatste Nieuws,NEWS,NaN,NaN


## Converting the rows to the correct Data Types

Parse "```date```" to datetime

In [6]:
gold_belga_press_articles_df['date'] = pd.to_datetime(gold_belga_press_articles_df['date'], errors='coerce')

Convert "```match_id```" and cheacking its format

In [7]:
print(gold_belga_press_articles_df['match_id'].dropna().unique()[:10])

['dkbsomyigazbberxvdmjq7wgk' 'dlnn04m3bhefdkjof0qemprmc'
 'dmcif0n7veawzigj8cd5bm904' 'dngvv4b04pq0170pqr3c73uvo'
 'dojq6igk3d5ol4c78rkm0ugb8' 'dpjf3ys82e8ogok1huyfd64no'
 'dqvzs325coup7j9iqyfcazmdw' 'drhnleuu8ae9hudku3cehgb9w'
 'dt091ishr3km0wuuhiatev9xw' 'du1wib350joxeh5hdv9v8yihg']


Since the "```match_id```" is a string, there's no need to convert it to another Data Type

checking if "```days_to_match```" should be int 

In [8]:
# Since it's float because of the NaN values, we can convert it to Int64
gold_belga_press_articles_df['days_to_match'] = (
    gold_belga_press_articles_df['days_to_match']
    .astype('Int64')  # capital I = pandas nullable integer
)

Converting "```type```" and "```source```" to categoricals for memory efficiency

In [9]:
gold_belga_press_articles_df['type'] = gold_belga_press_articles_df['type'].astype('category')
gold_belga_press_articles_df['source'] = gold_belga_press_articles_df['source'].astype('category')

In [10]:
gold_belga_press_articles_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48677 entries, 0 to 48676
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   article_id     48677 non-null  int64         
 1   date           48677 non-null  datetime64[ns]
 2   title          48677 non-null  object        
 3   lead           48667 non-null  object        
 4   source         48677 non-null  category      
 5   type           48677 non-null  category      
 6   match_id       8592 non-null   object        
 7   days_to_match  8592 non-null   Int64         
dtypes: Int64(1), category(2), datetime64[ns](1), int64(1), object(3)
memory usage: 2.4+ MB


## Dealing with NULL and duplicate values

In [11]:
gold_belga_press_articles_df['days_to_match'].describe()

count      8592.0
mean    -0.162244
std      1.000281
min          -2.0
25%          -1.0
50%           0.0
75%           1.0
max           1.0
Name: days_to_match, dtype: Float64


* -2, -1 → articles published 2 or 1 day before the match (build-up coverage)
* 0 → match day articles
* 1 → post-match articles (reactions, reports)

In [12]:
print("match day articles:")
print(gold_belga_press_articles_df.query('days_to_match == 0').shape[0])
print("post match articles:")
print(gold_belga_press_articles_df.query('days_to_match == 1').shape[0])
print("articles published 2 days before the match:")
print(gold_belga_press_articles_df.query('days_to_match == -2').shape[0])
print("articles published 1 day before the match:")
print(gold_belga_press_articles_df.query('days_to_match == -1').shape[0])

match day articles:
3421
post match articles:
2497
articles published 2 days before the match:
1217
articles published 1 day before the match:
1457


In [13]:
gold_belga_press_articles_df['days_to_match'].value_counts().sort_index()

days_to_match
-2    1217
-1    1457
0     3421
1     2497
Name: count, dtype: Int64

## Validating ```match_id``` join key

In [14]:
# Load the match table for comparison
gold_match_df = pd.read_csv('../raw/gold_match.csv', sep=',', on_bad_lines='skip')

# Check formats side by side
print("Articles match_id sample:", gold_belga_press_articles_df['match_id'].dropna().unique()[:5])
print("Match match_id sample:   ", gold_match_df['match_id'].unique()[:5])

# Check how many article match_ids actually exist in gold_match
linked_articles = gold_belga_press_articles_df['match_id'].dropna()
overlap = linked_articles.isin(gold_match_df['match_id'])
print(f"Matched: {overlap.sum()} / {len(linked_articles)} ({overlap.mean():.1%})")

Articles match_id sample: ['dkbsomyigazbberxvdmjq7wgk' 'dlnn04m3bhefdkjof0qemprmc'
 'dmcif0n7veawzigj8cd5bm904' 'dngvv4b04pq0170pqr3c73uvo'
 'dojq6igk3d5ol4c78rkm0ugb8']
Match match_id sample:    ['d0te6swsv2y99ywgugc2utbmc' 'd256yo3eng04m0fu7b4sl7wno'
 'd3pqkck2grzx98jg4sofhp8us' 'd4mn5ksbxuvnaww4pmommxhqs'
 'd5htdqmc8w72upys41sfxhfkk']
Matched: 8592 / 8592 (100.0%)


All the IDs are matching.

## Handling Missing Values

In [15]:
print(gold_belga_press_articles_df[gold_belga_press_articles_df['lead'].isna()][['source', 'type', 'date']])

# For NLP feature use, fill with empty string to avoid downstream errors
# gold_belga_press_articles_df['lead'] = gold_belga_press_articles_df['lead'].fillna('')

                                   source  type       date
4906   Economy - Other Belgian publishers  NEWS 2023-11-27
16869  Economy - Other Belgian publishers  NEWS 2023-08-07
27728                      Het Nieuwsblad  NEWS 2021-10-01
42054                  Het Laatste Nieuws  NEWS 2022-10-30
42055                  Het Laatste Nieuws  NEWS 2022-10-30
42056                  Het Laatste Nieuws  NEWS 2022-10-30
42057                  Het Laatste Nieuws  NEWS 2022-10-30
46851   Sports - Other Belgian publishers  NEWS 2026-01-22
46853   Sports - Other Belgian publishers  NEWS 2026-01-22
47202   Sports - Other Belgian publishers  NEWS 2026-02-02


For this project, there's no need to remove or fill in the NA values

## Deadling with Duplicate Values

Duplicate check

In [16]:
print("Duplicate rows:", gold_belga_press_articles_df.duplicated().sum())
print("Duplicate article_ids:", gold_belga_press_articles_df['article_id'].duplicated().sum())

Duplicate rows: 0
Duplicate article_ids: 0


Date Sanity Check

In [17]:
print("Date range:", gold_belga_press_articles_df['date'].min(), "→", gold_belga_press_articles_df['date'].max())
print("Coerced NaT count:", gold_belga_press_articles_df['date'].isna().sum())

Date range: 2021-03-13 00:00:00 → 2026-03-12 00:00:00
Coerced NaT count: 0


Pre-match aggregation

In [19]:
buzz_df = (
    gold_belga_press_articles_df[gold_belga_press_articles_df['days_to_match'].isin([-2, -1])]
    .groupby('match_id')
    .agg(pre_match_article_count=('article_id', 'count'))
    .reset_index()
)
buzz_df.head()
buzz_df.to_csv('../cleaned/match_buzz.csv', index=False)

This table shows all the total pre-match articles for each match
